# CRM Access Governance & Customer Data Protection
## Stage 10 — Governance Monitoring / Power BI

This notebook defines the full monitoring layer for the CRM Data Governance project.

The objective is to convert the analytical model into a practical **governance monitoring product** in Power BI.

### Main Goals

1. Define the Power BI semantic model.
2. Define core governance KPIs.
3. Create DAX measures.
4. Design dashboard pages.
5. Define visuals, filters, slicers, drill-through, and navigation.
6. Define executive storytelling.
7. Define Data Quality, Privacy, Access Governance, Metadata, Lineage, and Operating Model monitoring.
8. Prepare the model for final portfolio presentation.

> **Important:** Power BI-specific visual construction, relationship creation, bookmarks, and final formatting must be completed in Power BI Desktop. This notebook serves as the complete implementation specification and analytical design.


# 1. Power BI Model Overview

The recommended semantic model is based on the Stage 9 star schema.

```text
dim_role
dim_action
dim_device
dim_sensitivity
dim_rule
dim_risk
dim_hour
dim_region
dim_lead_source
        \
         \
       fact_access_event

dim_customer_privacy_safe
```

### Relationship Pattern

All access dimensions should filter `fact_access_event` using:

- `1:*`
- single-direction filtering
- dimension → fact

The privacy-safe customer dimension remains logically separate because the synthetic access dataset does not contain a governed customer foreign key.


# 2. Recommended Power BI Tables

Import the following Stage 9 outputs:

- `fact_access_event`
- `dim_role`
- `dim_action`
- `dim_device`
- `dim_sensitivity`
- `dim_rule`
- `dim_risk`
- `dim_hour`
- `dim_region`
- `dim_lead_source`
- `dim_customer_privacy_safe`

Optional governance tables:

- Data Quality Rule Catalog
- Privacy Control Catalog
- Metadata Catalog
- Governance Decision Log
- Issue Register
- RACI Matrix
- Lineage Mapping

These supporting tables can be used on governance-specific pages.


# 3. Core DAX Measures

The following measures form the foundation of the dashboard.


## 3.1 Total Access Events

```DAX
Total Access Events =
SUM ( fact_access_event[Access_Count] )
```


## 3.2 Original Blocked Access

```DAX
Original Blocked Access =
SUM ( fact_access_event[Is_Blocked] )
```


## 3.3 Original Block Rate

```DAX
Original Block Rate =
DIVIDE (
    [Original Blocked Access],
    [Total Access Events]
)
```


## 3.4 Proposed Blocked Access

```DAX
Proposed Blocked Access =
SUM ( fact_access_event[Proposed_Block_Flag] )
```


## 3.5 Proposed Block Rate

```DAX
Proposed Block Rate =
DIVIDE (
    [Proposed Blocked Access],
    [Total Access Events]
)
```


## 3.6 Access Under Review

```DAX
Access Under Review =
SUM ( fact_access_event[Review_Flag] )
```


## 3.7 Review Rate

```DAX
Review Rate =
DIVIDE (
    [Access Under Review],
    [Total Access Events]
)
```


## 3.8 Allowed Access

```DAX
Allowed Access =
SUM ( fact_access_event[Allow_Flag] )
```


## 3.9 Allow Rate

```DAX
Allow Rate =
DIVIDE (
    [Allowed Access],
    [Total Access Events]
)
```


## 3.10 Critical Risk Events

```DAX
Critical Risk Events =
CALCULATE (
    [Total Access Events],
    dim_risk[Contextual_Risk_Level] = "CRITICAL"
)
```


## 3.11 High + Critical Risk Events

```DAX
High or Critical Risk Events =
CALCULATE (
    [Total Access Events],
    dim_risk[Contextual_Risk_Level] IN { "HIGH", "CRITICAL" }
)
```


## 3.12 Average Contextual Risk Score

```DAX
Average Contextual Risk Score =
AVERAGE ( fact_access_event[Contextual_Risk_Score] )
```


## 3.13 Average Anomaly Score

```DAX
Average Anomaly Score =
AVERAGE ( fact_access_event[Anomaly_Score] )
```


## 3.14 Average Governance Score

```DAX
Average Governance Score =
AVERAGE ( fact_access_event[Governance_Score] )
```


## 3.15 DQ Pass Records

```DAX
DQ Pass Records =
CALCULATE (
    COUNTROWS ( fact_access_event ),
    fact_access_event[DQ_Status] = "PASS"
)
```


## 3.16 DQ Pass Rate

```DAX
DQ Pass Rate =
DIVIDE (
    [DQ Pass Records],
    [Total Access Events]
)
```


## 3.17 DQ Failed Records

```DAX
DQ Failed Records =
CALCULATE (
    COUNTROWS ( fact_access_event ),
    fact_access_event[DQ_Status] <> "PASS"
)
```


## 3.18 Average Failed DQ Rules

```DAX
Average Failed DQ Rules =
AVERAGE ( fact_access_event[Failed_Rule_Count] )
```


## 3.19 BYOD Access Events

```DAX
BYOD Access Events =
CALCULATE (
    [Total Access Events],
    dim_device[Is_BYOD] = TRUE()
)
```


## 3.20 High Sensitivity Access

```DAX
High Sensitivity Access =
CALCULATE (
    [Total Access Events],
    dim_sensitivity[High_Sensitivity_Flag] = TRUE()
)
```


# 4. Dashboard Page Architecture

Recommended page sequence:

1. **Governance Overview**
2. **Access Governance**
3. **Risk & Security**
4. **Data Quality**
5. **Privacy Monitoring**
6. **Metadata & Lineage**
7. **Governance Operations**

This structure reflects the entire governance lifecycle developed in Stages 1–9.


# 5. Page 1 — Governance Overview

### Objective
Provide an executive-level summary of the governance environment.

### KPI Cards
- Total Access Events
- Proposed Block Rate
- Review Rate
- Critical Risk Events
- DQ Pass Rate
- High Sensitivity Access

### Recommended Visuals

**1. Access Decision Distribution**
- visual: donut or stacked bar
- values:
  - ALLOW
  - REVIEW
  - BLOCK

**2. Risk Level Distribution**
- visual: column chart
- axis: `Contextual_Risk_Level`
- value: Total Access Events

**3. Access by Role**
- visual: horizontal bar chart
- axis: Role
- value: Total Access Events

**4. Governance Rule Triggers**
- visual: bar chart
- axis: Rule_ID
- value: access events

**5. Governance Summary Matrix**
- Role
- Proposed Block Rate
- Review Rate
- Average Contextual Risk Score
- DQ Pass Rate


# 6. Page 2 — Access Governance

### Objective
Monitor permissions, role-action controls, and proposed governance decisions.

### KPI Cards
- Proposed Blocked Access
- Access Under Review
- Allowed Access
- Proposed Block Rate

### Recommended Visuals

**Role × CRM Action Matrix**
- rows: Role
- columns: CRM_Action
- values: Proposed Block Rate

This becomes one of the central visuals in the project.

**Access Decision by Role**
- stacked bar chart
- axis: Role
- legend: Proposed_Access_Decision
- value: Access Events

**Access Decision by Action**
- stacked column chart
- axis: CRM_Action
- legend: Proposed_Access_Decision

**Rule Trigger Frequency**
- bar chart
- Rule_ID
- event count

**Permission vs. Proposed Decision**
- matrix or stacked chart


# 7. Page 3 — Risk & Security

### Objective
Monitor contextual risk and anomalous access behavior.

### KPI Cards
- Critical Risk Events
- High or Critical Risk Events
- Average Anomaly Score
- Average Governance Score
- BYOD Access Events

### Recommended Visuals

**Risk Level Distribution**
- LOW / MEDIUM / HIGH / CRITICAL

**Risk by Device**
- Device_Type × Contextual_Risk_Level

**Risk by Hour**
- line chart
- Access_Hour
- Average Contextual Risk Score

**Anomaly Score by Access Decision**
- boxplot if custom visual available
- otherwise clustered distribution / percentile chart

**High Sensitivity × Device**
- matrix
- Data_Sensitivity
- Device_Type
- Proposed Block Rate


# 8. Page 4 — Data Quality

### Objective
Monitor the health of data used by governance decisions.

### KPI Cards
- DQ Pass Rate
- DQ Failed Records
- Average Failed DQ Rules
- Critical DQ Failures

### Recommended Visuals

**DQ Status Distribution**
- PASS
- WARNING
- FAIL

**DQ Failures by Role**
- bar chart

**DQ Failures by CRM Action**
- bar chart

**Failed Rule Count Distribution**
- column chart

**Data Quality Matrix**
- Role
- Access Events
- DQ Pass Rate
- Average Failed DQ Rules

If the Data Quality Rule Catalog is imported:

**DQ Rules Table**
- Rule ID
- Dimension
- Severity
- Pass Rate
- Fail Rate


# 9. Page 5 — Privacy Monitoring

### Objective
Demonstrate privacy-aware CRM analytics.

### KPI Cards
- Customer Records
- Marketing Consent Rate
- Restricted Fields
- Direct Identifier Fields
- Analytics-Safe Fields

### Recommended Visuals

**Customer Segment Distribution**
- bar chart

**Marketing Consent**
- donut chart

**Customers by State**
- map or bar chart

**Revenue by Customer Segment**
- column chart

**Privacy Classification Inventory**
- Internal
- Confidential
- Restricted

**Privacy Controls Table**
- Control ID
- Type
- Field
- Treatment
- Rationale


# 10. Page 6 — Metadata & Lineage

### Objective
Show that governance extends beyond access controls.

### KPI Cards
- Catalog Coverage %
- CDE Count
- Lineage Coverage %
- Registered Transformations
- Registered Analytical Outputs

### Recommended Visuals

**Fields by Data Domain**
- bar chart

**Critical Data Elements by Domain**
- stacked bar

**Fields by Sensitivity**
- column chart

**Lineage Coverage**
- gauge or KPI card

**Source-to-Target Mapping**
- table:
  - Source Field
  - Transformation
  - Target Field
  - Governance Rule
  - KPI

**Rule-to-KPI Impact Analysis**
- table or decomposition tree


# 11. Page 7 — Governance Operations

### Objective
Monitor governance as an operating process.

### KPI Cards
- Open Governance Issues
- Critical Open Issues
- Active Policy Exceptions
- Overdue Issues
- Rules Pending Review

### Recommended Visuals

**Issues by Category**
- DQ
- Privacy
- Access
- Metadata
- Lineage
- KPI
- Policy

**Issues by Severity**
- Low
- Medium
- High
- Critical

**Issues by Status**
- Open
- Investigating
- Remediating
- Resolved
- Exception

**Governance Forums**
- table:
  - Forum
  - Cadence
  - Participants
  - Focus

**Decision Log**
- table:
  - Decision ID
  - Type
  - Decision
  - Owner
  - Status


# 12. Global Slicers

Recommended global slicers:

- Role
- CRM Action
- Region
- Device Type
- Data Sensitivity
- Risk Level
- Proposed Access Decision
- Governance Rule
- DQ Status

Use synchronized slicers only where cross-page consistency is useful.


# 13. Drill-Through Strategy

Recommended drill-through page:

## Access Event Detail

Fields:
- User_ID
- Role
- CRM_Action
- Device_Type
- Data_Sensitivity
- Permission_Granted
- Contextual_Risk_Score
- Contextual_Risk_Level
- Anomaly_Score
- Governance_Score
- Proposed_Access_Decision
- Triggered_Rule_ID
- Failed_Rule_Count
- DQ_Status

This page allows a governance analyst to move from aggregate monitoring to event-level investigation.


# 14. Tooltip Strategy

Custom report-page tooltips can provide additional context without cluttering pages.

Examples:

### Role Tooltip
- Access Events
- Proposed Block Rate
- Review Rate
- Average Risk Score

### Rule Tooltip
- Rule Description
- Rule Category
- Rule Priority
- Trigger Count

### Data Sensitivity Tooltip
- Access Events
- Block Rate
- BYOD Events
- Average Anomaly Score


# 15. Navigation

Recommended navigation:

```text
Overview
   ↓
Access Governance
   ↓
Risk & Security
   ↓
Data Quality
   ↓
Privacy
   ↓
Metadata & Lineage
   ↓
Governance Operations
```

Use page navigation buttons or a consistent side/top navigation menu.


# 16. Executive Storytelling

The dashboard should answer a sequence of questions:

### 1. What is happening?
How many access events exist and how many are blocked or reviewed?

### 2. Where is the risk?
Which roles, actions, devices, sensitivity levels, and contexts generate the highest risk?

### 3. Can the data supporting those decisions be trusted?
What is the Data Quality status?

### 4. Are customer data protected?
Which fields are restricted, minimized, masked, or pseudonymized?

### 5. Can we trace the decision?
Which source fields, rules, controls, and transformations produced the outcome?

### 6. Who owns remediation?
Which governance role, steward, or committee is accountable?


# 17. Recommended Visual Hierarchy

Each page should follow the same pattern:

```text
Page Title
↓
Executive KPI Cards
↓
Primary Diagnostic Visual
↓
Supporting Breakdown Visuals
↓
Detailed Matrix / Table
```

This keeps the report readable for both executives and governance analysts.


# 18. Formatting Guidelines

Recommended principles:

- use consistent naming across notebooks and Power BI;
- avoid excessive decorative visuals;
- highlight exceptions and risk rather than all data equally;
- keep KPI definitions visible through tooltips or documentation;
- use the same decision labels everywhere:
  - ALLOW
  - REVIEW
  - BLOCK
- use consistent risk ordering:
  - LOW
  - MEDIUM
  - HIGH
  - CRITICAL
- preserve single-direction star-schema filtering unless a specific need justifies otherwise.


# 19. Power BI Validation Checklist

Before finalizing the report:

### Model
- [ ] All dimensions have unique keys.
- [ ] Fact foreign keys resolve correctly.
- [ ] Relationships are 1:*.
- [ ] Filter direction is single.
- [ ] No unnecessary many-to-many relationship exists.

### Measures
- [ ] Total Access Events matches fact row count.
- [ ] Proposed Block Rate matches Python validation.
- [ ] Review Rate matches Python validation.
- [ ] DQ Pass Rate matches Python validation.
- [ ] Critical Risk Event count matches Python validation.

### Report
- [ ] Slicers filter expected visuals.
- [ ] Drill-through works.
- [ ] Tooltips show correct context.
- [ ] Page navigation works.
- [ ] Titles and KPI names are consistent.


# 20. Portfolio Positioning

This dashboard should not be presented merely as a Power BI report.

The stronger framing is:

> **A governance monitoring solution for CRM access, data quality, privacy, metadata, lineage, and operational governance.**

The Power BI layer is the monitoring interface for a broader governance framework created across the previous stages.


# 21. Stage 10 Deliverable

At the end of this stage, the portfolio should contain:

- the Stage 9 analytical model;
- the Power BI semantic model;
- all core DAX measures;
- seven governance monitoring pages;
- drill-through event analysis;
- governance KPI definitions;
- executive storytelling;
- monitoring logic covering:
  - access;
  - risk;
  - Data Quality;
  - privacy;
  - metadata;
  - lineage;
  - governance operations.


# 22. Next Step — AI Governance Extension

## Stage 11 — AI Governance

The project will next simulate the introduction of an AI assistant connected to CRM data.

Planned topics:

- AI use-case inventory;
- AI risk classification;
- model and use-case ownership;
- CRM data access by AI;
- PII exposure risks;
- prompt and response governance;
- human oversight;
- AI access controls;
- logging and traceability;
- AI governance control catalog;
- initial mapping to NIST AI RMF / Responsible AI concepts.

This will extend the project from **Data Governance** into **Data & AI Governance**.
